In [12]:
%load_ext autoreload
%autoreload 2
import pandas as pd
import numpy as np
import tushare as ts
import edhec_risk_kit as erk 
import time
import cvxpy as cp

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
w=erk.read_hs300_weights()

In [14]:
returns=erk.read_returns()
w=erk.read_hs300_weights()
w=w.iloc[-2]
w=w.dropna()
w=w.sort_values(ascending=False)
id=w.index
returns=returns.loc["2021-9":"2026-09"]
miss=returns.isna().mean()
returns=returns.loc[:,miss<=0.05]
codes=w.index[w.index.isin(returns.columns)]
codes=codes[:50]
codes
returns=returns[codes]
returns

,300308.SZ,600519.SH,300750.SZ,601318.SH,300502.SZ,600036.SH,601899.SH,603259.SH,000333.SZ,600900.SH,...,000858.SZ,000338.SZ,601088.SH,600309.SH,300476.SZ,600000.SH,300274.SZ,002415.SZ,002142.SZ,603019.SH
trade_date,,,,,,,,,,,,,,,,,,,,,
2021-09-01,0.007340,0.041085,-0.012448,0.031864,0.015465,0.037377,-0.037804,-0.000151,0.047727,0.024819,...,0.029848,-0.019432,-0.023500,0.002812,0.026968,0.014365,-0.096579,0.035214,0.032508,0.008475
2021-09-02,0.000561,-0.001979,0.009801,0.001165,-0.017529,0.015948,0.019645,-0.032083,0.000578,0.008073,...,-0.031197,0.008638,0.029698,-0.017757,0.008471,0.005447,0.005701,-0.033828,0.038681,-0.012605
2021-09-03,0.037255,0.024351,-0.014184,0.003104,0.030126,0.007752,0.015596,0.040305,0.012285,0.004004,...,0.023158,-0.027708,-0.006962,-0.021408,0.000840,0.001083,-0.021274,-0.000195,0.001155,0.004255
2021-09-06,-0.002971,0.007508,0.060843,-0.002514,0.034923,-0.015385,-0.010840,0.051907,0.029697,-0.001994,...,0.027053,0.031088,0.040060,0.016918,0.140999,-0.003247,0.002360,0.039494,0.000577,0.021540
2021-09-07,-0.024648,-0.001000,-0.004050,0.014734,-0.002469,0.031445,0.002740,0.003982,0.012479,0.000500,...,0.002412,-0.001005,0.024555,0.023329,-0.019860,0.014115,0.012911,-0.009545,0.011527,0.005531
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-09-23,-0.005627,-0.002042,-0.011916,-0.010834,-0.008352,-0.005390,0.000000,0.000596,-0.000848,0.002141,...,-0.002835,-0.042175,-0.006314,0.013303,0.009087,-0.006637,-0.015533,-0.002110,0.000000,-0.013818
2026-09-24,-0.028878,-0.011381,-0.024917,-0.011509,-0.035904,0.002217,-0.044572,-0.033046,0.002183,0.009972,...,-0.009525,-0.022789,0.017369,-0.030028,-0.047524,0.002227,-0.030174,-0.014497,0.005472,-0.015928
2026-09-28,-0.090260,0.005562,-0.005145,-0.015962,-0.081149,-0.001229,-0.019660,-0.003695,-0.007864,0.006700,...,-0.010909,-0.026877,0.007495,-0.000720,-0.082470,0.017778,-0.037406,-0.014404,-0.010312,-0.026165


In [15]:
n=returns.shape[1]
n
cov=returns.cov()
cov
w=cp.Variable(n)
constraints=[]
equal=cp.sum(w)==1
high=w<=0.05
low=w>=0
constraints.append(equal)
constraints.append(low)
problem=cp.Problem(cp.Minimize(cp.quad_form(w,cov)),constraints)
problem.solve()
problem.status
w_cp=pd.Series(w.value,index=returns.columns)
w_cp_see=w_cp.clip(0)
w_cp_see=w_cp_see.round(2).sort_values(ascending=False)
w_cp_see
q_returns=returns@w_cp
q_returns.mean()

np.float64(0.0005720574500090822)

In [16]:

returns=returns.dropna()
cov=returns.cov()
w=erk.cp_weight(cov,mu=returns,target=0.1)
w_cp=pd.Series(w,index=returns.columns)
w_cp_see=w_cp.clip(0)
w_cp_see=w_cp_see.round(2).sort_values(ascending=False)
w_cp_see
w_cp_see
q_returns=returns@w_cp
q_returns.mean()

np.float64(0.0005498641530634177)